<a href="https://colab.research.google.com/github/fs0440250-cmyk/Detective-de-se-ales/blob/main/Proyecto_Detective_Senales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
Santana Maldonado Felipeimport numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# ==========================================
# 1.2. PARÁMETROS DE LA SEÑAL
# ==========================================
fs = 1000.0       # Frecuencia de muestreo (Hz)
fc = 100.0        # Frecuencia portadora (Hz)
Rs = 10.0         # Tasa de símbolos (símbolos por segundo)
samples_per_symbol = int(fs / Rs)
num_symbols = 10  # Número de símbolos por cada muestra de señal
signal_length = samples_per_symbol * num_symbols

num_samples_per_class = 1000  # 1,000 muestras por tipo de modulación

# ==========================================
# 1.3 Y 1.4. PROGRAMAR GENERADORES Y CREAR DATASET
# ==========================================
def generate_modulated_signal(mod_type):
    # Generar bits o símbolos aleatorios según la modulación
    if mod_type == 'ASK':
        # Amplitud variable (0 o 1)
        data = np.random.randint(0, 2, num_symbols)
        symbol_signal = np.repeat(data, samples_per_symbol)
        t = np.arange(len(symbol_signal)) / fs
        signal = symbol_signal * np.sin(2 * np.pi * fc * t)

    elif mod_type == 'FSK':
        # Frecuencia variable entre dos tonos
        data = np.random.randint(0, 2, num_symbols)
        f1, f2 = fc - 20, fc + 20
        freqs = np.where(data == 0, f1, f2)
        symbol_signal = np.repeat(freqs, samples_per_symbol)
        t = np.arange(len(symbol_signal)) / fs
        signal = np.sin(2 * np.pi * symbol_signal * t)

    elif mod_type == 'PSK':
        # Fase variable (0 o pi)
        data = np.random.randint(0, 2, num_symbols)
        phases = data * np.pi
        symbol_signal = np.repeat(phases, samples_per_symbol)
        t = np.arange(len(symbol_signal)) / fs
        signal = np.sin(2 * np.pi * fc * t + symbol_signal)

    elif mod_type == 'QAM':
        # 16-QAM simplificado (amplitud y fase combinadas en 4 niveles)
        data_i = np.random.choice([-3, -1, 1, 3], num_symbols)
        data_q = np.random.choice([-3, -1, 1, 3], num_symbols)
        i_signal = np.repeat(data_i, samples_per_symbol)
        q_signal = np.repeat(data_q, samples_per_symbol)
        t = np.arange(len(i_signal)) / fs
        signal = i_signal * np.cos(2 * np.pi * fc * t) - q_signal * np.sin(2 * np.pi * fc * t)

    else:
        raise ValueError("Modulación no soportada")

    # Normalizar la señal para mantener estabilidad numérica
    signal = signal / (np.max(np.abs(signal)) + 1e-8)
    return signal

# ==========================================
# 1.5. AÑADIR RUIDO (AWGN CON DIFERENTES SNR)
# ==========================================
def add_awgn_noise(signal, snr_db):
    # Calcular potencia de la señal
    signal_power = np.mean(signal ** 2)
    # Calcular potencia del ruido basada en el SNR deseado en dB
    snr_linear = 10 ** (snr_db / 10.0)
    noise_power = signal_power / snr_linear
    # Generar ruido gaussiano blanco
    noise = np.sqrt(noise_power) * np.random.normal(0, 1, len(signal))
    return signal + noise

# Construcción de las bases de datos
modulations = ['ASK', 'FSK', 'PSK', 'QAM']
X = []
y = []

print("Generando dataset...")
for label_idx, mod in enumerate(modulations):
    for _ in range(num_samples_per_class):
        # Generar señal limpia
        clean_signal = generate_modulated_signal(mod)

        # Seleccionar aleatoriamente un nivel de SNR entre 0 dB y 30 dB (simulando canal real)
        snr_db = np.random.uniform(0, 30)
        noisy_signal = add_awgn_noise(clean_signal, snr_db)

        X.append(noisy_signal)
        y.append(label_idx)

X = np.array(X)
y = np.array(y)

print(f"Dataset generado con éxito. Forma de X (datos): {X.shape}, Forma de y (etiquetas): {y.shape}")

# ==========================================
# 1.6. SEPARACIÓN DE DATOS (ENTRENAMIENTO 80% Y PRUEBA 20%)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba: {X_test.shape[0]} muestras")

Generando dataset...
Dataset generado con éxito. Forma de X (datos): (4000, 1000), Forma de y (etiquetas): (4000,)
Datos de entrenamiento: 3200 muestras
Datos de prueba: 800 muestras


In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

# ==========================================
# 2.2. DISEÑAR EL MODELO (CNN 1D)
# ==========================================
# Obtenemos la longitud de la señal y el número de canales a partir de los datos de entrenamiento
input_shape = (X_train.shape[1], 1)

model = Sequential([
    # Capa de entrada y primera capa convolucional
    Conv1D(filters=32, kernel_size=5, activation='relu', input_shape=input_shape),
    MaxPooling1D(pool_size=2),

    # Segunda capa convolucional para extraer patrones más complejos
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),

    # Aplanar las características para pasarlas a la capa densa
    Flatten(),

    # Capa oculta con Dropout para evitar sobreajuste (overfitting)
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Capa de salida Densa con Softmax (4 neuronas, una para cada modulación: ASK, FSK, PSK, QAM)
    Dense(4, activation='softmax')
])

# ==========================================
# 2.3. COMPILAR EL MODELO
# ==========================================
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Mostrar la arquitectura del modelo en pantalla
model.summary()

# ==========================================
# 2.4. ENTRENAR EL MODELO (.fit)
# ==========================================
# Redimensionamos los datos para que tengan la forma correcta requerida por Conv1D (Muestras, Longitud, 1 Canal)
X_train_cnn = np.expand_dims(X_train, axis=-1)
X_test_cnn = np.expand_dims(X_test, axis=-1)

history = model.fit(
    X_train_cnn, y_train,
    epochs=15,             # Número de pasadas completas por el dataset
    batch_size=32,         # Tamaño de lote por iteración
    validation_split=0.2   # 20% de los datos de entrenamiento para validación interna por época
)

# ==========================================
# 2.5. GUARDAR EL MODELO
# ==========================================
model.save('modelo_detector_senales.h5')
print("\n¡Modelo entrenado y guardado exitosamente como 'modelo_detector_senales.h5'!")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 996, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 498, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 494, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 247, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 15808)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,011,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,022,532 (3.90 MB)

 Trainable params: 1,022,532 (3.90 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 68ms/step - accuracy: 0.8871 - loss: 0.2937 - val_accuracy: 0.9937 - val_loss: 0.0172
Epoch 2/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 6s 80ms/step - accuracy: 0.9965 - loss: 0.0207 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 3/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 0.9945 - loss: 0.0187 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 4/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.9973 - loss: 0.0085 - val_accuracy: 0.9984 - val_loss: 0.0047
Epoch 5/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.9996 - loss: 0.0029 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 6/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 9s 107ms/step - accuracy: 0.9984 - loss: 0.0059 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 7/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 0.9984 - loss: 0.0096 - val_accuracy: 0.9969 - val_loss: 0.0068
Epoch 8/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 6s 81ms/step - accuracy: 0.9996 - loss: 0.0030 - val_accuracy: 0.9937 - 


¡Modelo entrenado y guardado exitosamente como 'modelo_detector_senales.h5'!
